# Run HRM-Text-1B with KerasHub

This is a practical, GPU-backed introduction to the proposed KerasHub HRM-Text port. It converts the official Apache-2.0 `sapientinc/HRM-Text-1B` checkpoint, prepares a PrefixLM prompt, and generates a short continuation. A Colab T4 is sufficient.

> **Current status.** This implementation is under review and does not yet have an upstream hosted preset. The notebook intentionally checks out one immutable source commit and converts the official checkpoint locally. Once a maintained preset is published, loading it will be a one-line `from_preset()` call.

## What makes this model different?

HRM-Text runs shared high-level (H) and low-level (L) Transformer stacks repeatedly. The official 1B configuration has 16 layers per stack, two H cycles, and three L cycles. During generation, each logical stack invocation needs its own KV-cache slot: `16 × 2 × (3 + 1) = 128`. The public API remains familiar: provide token IDs and masks, then call `generate()`.

In [ ]:
import json
import os
import subprocess

# An immutable source revision makes this notebook reproducible. Update this
# value whenever a new implementation commit is pushed.
REPOSITORY = "https://github.com/pzarzycki/keras-hub.git"
KERAS_HUB_REVISION = "5ff84000167698f0c412fb9cb8121c878e932749"
MODEL_ID = "sapientinc/HRM-Text-1B"
MODEL_REVISION = "9f082d68b8cd0ebc56e33f1c88c45609174c272c"
WORKDIR = "/content/keras-hub"

subprocess.run(["bash", "-lc", "command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
subprocess.run(["git", "clone", REPOSITORY, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "checkout", "--detach", KERAS_HUB_REVISION], check=True)
# Pin Transformers without globally upgrading Colab's compiled scientific stack.
subprocess.run(["uv", "pip", "install", "--system", "-q", "transformers==5.9.0", "safetensors"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", WORKDIR], check=True)

# Select the Keras backend before importing Keras. The conversion below uses
# PyTorch because it reads the original checkpoint directly on the GPU.
os.environ["KERAS_BACKEND"] = "torch"


In [ ]:
import os
os.chdir(WORKDIR)

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")

torch.set_default_device("cuda")
import keras
from keras_hub.src.models.hrm_text.hrm_text_causal_lm import HrmTextCausalLM
from tools.checkpoint_conversion.convert_hrm_text_checkpoints import (
    convert_tokenizer,
    convert_weights,
    create_backbone,
)

keras.config.set_dtype_policy("float32")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32
).cuda().eval()

backbone = create_backbone(hf_model.config)
convert_weights(backbone, hf_model)
tokenizer = convert_tokenizer(hf_tokenizer)
model = HrmTextCausalLM(backbone, preprocessor=None)
print(f"Keras backend: {keras.config.backend()}")
print(f"Logical KV-cache slots: {backbone.cache_slots}")


## Build a PrefixLM prompt

The released model is a base pre-alignment model, not an instruction-tuned chat model. Its upstream prompt format begins with `<|im_start|>`, optionally contains condition tokens, and closes with `<|im_end|>`. The condition names below are convenience labels for the published special tokens; use them as a starting point rather than a quality guarantee.

In [ ]:
CONDITIONS = {
    "direct": "<|object_ref_start|>",
    "cot": "<|object_ref_end|>",
    "noisy": "<|quad_start|>",
    "synth": "<|quad_end|>",
}

def make_prompt(text, conditions=("synth", "cot")):
    """Return an upstream-format PrefixLM prompt."""
    condition_tokens = "".join(CONDITIONS[name] for name in conditions)
    return f"<|im_start|>{condition_tokens}{text}<|im_end|>"

prompt = make_prompt("What is the capital of France?")
print(prompt)


In [ ]:
def generate(text, conditions=("synth", "cot"), max_new_tokens=4):
    """Generate a short continuation using the model's cached decode path."""
    prompt = make_prompt(text, conditions)
    prompt_ids = hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    max_length = len(prompt_ids) + max_new_tokens
    token_ids = np.full((1, max_length), tokenizer.pad_token_id, dtype="int32")
    token_ids[0, : len(prompt_ids)] = prompt_ids
    padding_mask = np.zeros((1, max_length), dtype="int32")
    padding_mask[0, : len(prompt_ids)] = 1

    # All present tokens are PrefixLM context; KerasHub expands its logical
    # 128-slot cache as generation proceeds.
    model.compile(sampler="greedy")
    generated = model.generate(
        {
            "token_ids": token_ids,
            "padding_mask": padding_mask,
            "token_type_ids": padding_mask.copy(),
        },
        max_length=max_length,
        stop_token_ids=None,
    )
    return str(tokenizer.detokenize(generated["token_ids"][0]))

first_generation = generate("What is the capital of France?")
print(first_generation)


In [ ]:
# Try a different published condition. Increase max_new_tokens gradually: each
# generated token exercises the full recurrent 128-slot cache.
second_generation = generate("Complete: The Pacific Ocean is", conditions=("direct",), max_new_tokens=4)
print(second_generation)
with open("/content/hrm_text_1b_usage_results.json", "w") as f:
    json.dump({"first_generation": first_generation, "second_generation": second_generation, "cache_slots": backbone.cache_slots}, f)


## PrefixLM training inputs

For training, use `HrmTextCausalLMPreprocessor` with separate `prefix` and `response` fields. Prefix tokens are context only; response tokens receive loss weight. This cell shows the shape of one example without starting a training job.

In [ ]:
from keras_hub.src.models.hrm_text.hrm_text_causal_lm_preprocessor import (
    HrmTextCausalLMPreprocessor,
)

preprocessor = HrmTextCausalLMPreprocessor(tokenizer, sequence_length=32)
features, labels, sample_weight = preprocessor(
    {
        "prefix": ["<|im_start|><|object_ref_start|>Question: 2 + 2 ="],
        "response": [" 4<|im_end|>"],
    }
)
print("token_type_ids (1=prefix):", keras.ops.convert_to_numpy(features["token_type_ids"])[0])
print("loss weights (1=response):", keras.ops.convert_to_numpy(sample_weight)[0])


## Save locally and next steps

Conversion is deliberately explicit in this review-stage notebook. If you need to reuse the converted model in the same environment, save it as a local Keras preset. Do not treat this base checkpoint as a chat model or as an evaluation benchmark. When an upstream hosted preset is available, prefer that maintained artifact over a personal checkpoint copy.

In [ ]:
# model.save_to_preset("/content/hrm_text_1b")
# Later, load the local directory with:
# reloaded = HrmTextCausalLM.from_preset("/content/hrm_text_1b")
